# 01 - Give the badge one key, then test the door

**Goal:** prove that a real Entra child agent identity can call a protected MCP tool, while forbidden calls are rejected. No model or Agent 365 license is needed.

**Before starting:** finish [00 - Identity setup](./00_Identity_Setup.ipynb). You need the tenant, child client ID, blueprint client ID, and private blueprint credential. The administrator also needs the child's **Object ID** for grants.

**MCP** is the way our code talks to tools. We run a small MCP server on this computer. It checks Entra's signed token before it returns anything. This is our custom teaching API, not Microsoft Work IQ.

Steps 1-2 are one-time administrator setup. Steps 3-4 are the customer demo.

**New to scopes and roles?** Read [the plain-language workshop guide](./WORKSHOP_START_HERE.md): direct access is the agent's own authorization; OBO is delegated authorization for a signed-in user.

## 1. Register our tool API - once, administrator

1. Entra > **App registrations > New registration**: name `agent365-workshop-mcp-api`; choose single tenant; no redirect URI.
2. Copy its **Application/client ID** to `MCP_WORKSHOP_API_ID` in the private `.env`. Do not add a secret.
3. **Expose an API > Application ID URI**: set `api://<MCP-API-CLIENT-ID>`.
4. **Add a scope**: name `Workshop.Read`, **Admins only**, enabled. Consent text: `Read workshop identity`. This is the key used later when a user is involved.
5. **App roles > Create app role**: display name `Read workshop identity (application)`, allowed member types **Applications**, value `Workshop.Read.Application`, enabled. This is the key for the agent acting alone. Keep your existing delegated scope `Workshop.Read`; do not rename or delete it.
6. **Manifest**: set `api.requestedAccessTokenVersion` to `2`; preserve everything else.

**Why:** the delegated scope `Workshop.Read` and application role `Workshop.Read.Application` use distinct values to avoid Entra duplicate-value errors. The signed token contains `scp: Workshop.Read` for OBO or `roles: [Workshop.Read.Application]` for agent-only access. The server checks the matching permission before exposing tools and again on each call. The word Read does not automatically forbid writes: these tools only return identity metadata and contain no write operation. A future write tool must require its own write permission. Registering an API does not deploy a server; our Python helper starts it later.

## 2. Assign the key to the child - once, administrator

**Setup needed:** PowerShell 7, Microsoft.Graph.Applications module, and an administrator allowed to grant application permissions. Run in a terminal, NOT as a Python cell. Replace the three placeholders using `A365_TENANT_ID`, `A365_OBSERVABILITY_AGENT_ID` (child CLIENT ID), and `MCP_WORKSHOP_API_ID` from your private `.env`. Azure subscription access alone is not enough. Do not display the whole environment file.

This uses device-code sign-in: open the address it prints and enter its temporary code yourself. If an earlier WAM window is hidden, cancel with Ctrl+C and start in a fresh standalone PowerShell window. If authentication still fails, stop; see [sign-in troubleshooting](./LIVE_SETUP.md). The script resolves the child's Object ID instead of assuming it equals the client ID.

```powershell
$ErrorActionPreference = 'Stop'
Connect-MgGraph -TenantId '<TENANT-ID>' -Scopes 'Application.Read.All','AppRoleAssignment.ReadWrite.All' -UseDeviceAuthentication -ContextScope Process -ErrorAction Stop
$childClientId = '<CHILD-CLIENT-ID>'
$apiClientId = '<MCP-API-CLIENT-ID>'
$children = @(Get-MgServicePrincipal -Filter "appId eq '$childClientId'" -Property Id,DisplayName)
if ($children.Count -ne 1) { throw 'Expected one child agent identity in this tenant.' }
$childObjectId = $children[0].Id
$apis = @(Get-MgServicePrincipal -Filter "appId eq '$apiClientId'" -Property Id,AppRoles)
if ($apis.Count -ne 1) { throw 'Expected one MCP API principal; check Enterprise applications.' }
$api = $apis[0]
$roles = @($api.AppRoles | Where-Object { $_.Value -eq 'Workshop.Read.Application' -and $_.IsEnabled -and $_.AllowedMemberTypes -contains 'Application' })
if ($roles.Count -ne 1) { throw 'Expected one enabled Workshop.Read.Application role.' }
$role = $roles[0]
$existing = @(Get-MgServicePrincipalAppRoleAssignment -ServicePrincipalId $childObjectId -All | Where-Object { $_.ResourceId -eq $api.Id -and $_.AppRoleId -eq $role.Id })
if ($existing.Count -eq 0) {
    New-MgServicePrincipalAppRoleAssignment -ServicePrincipalId $childObjectId -PrincipalId $childObjectId -ResourceId $api.Id -AppRoleId $role.Id
}
$verified = @(Get-MgServicePrincipalAppRoleAssignment -ServicePrincipalId $childObjectId -All | Where-Object { $_.ResourceId -eq $api.Id -and $_.AppRoleId -eq $role.Id })
if ($verified.Count -eq 0) { throw 'Role assignment could not be verified; do not continue yet.' }
Write-Host "VERIFIED: $($children[0].DisplayName) has Workshop.Read.Application on this MCP API."
```

If the API principal is absent, an administrator must create it for the existing API app (not register another API). Allow the grant time to propagate, then obtain a fresh token.

**Why:** creating the role offers a key; assigning it gives that key to the **child**, not only to its blueprint. Client IDs are used to request tokens; object IDs are used in these directory grants. This app role does not grant OBO access; notebook 03 grants the separate delegated scope.

## 3. Load the badge and get a real token - each session

**Setup needed:** steps 1-2 plus notebook 00; use the project kernel. Start from the project root or `Agent_365`. Restart the kernel after editing `.env`.

The first cell loads settings without printing the secret. The second asks Entra for an MCP-specific token. Think of a token as a short-lived, signed pass. We never print it or send the blueprint secret to MCP.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().parent if Path.cwd().name == 'Agent_365' else Path.cwd()
sys.path.insert(0, str(root))
from Agent_365.mcp_workshop import MCPSettings, sign_in_agent, explore_mcp_identity

settings = MCPSettings.from_env()
print('Child client ID:', settings.agent_id)
print('Destination MCP API:', settings.api_id)

In [ ]:
access = await sign_in_agent(settings)

**What just happened:** blueprint credential + child ID (`fmi_path`) -> parent exchange token -> child token for `api://<MCP-API-ID>/.default`. `/.default` uses the permissions already granted; it does not grant new ones.

**Expected:** `Acquired app-only (agent) token`. If permission is missing, stop and fix step 2. We do not invent a token or use your user identity instead.

**If token routing fails:** first check the MCP API app's **Manifest > api.requestedAccessTokenVersion = 2** and save. Configure the API receiving the token, not the Graph browser client, blueprint, or child. A v2 token endpoint can still return a v1 access token when the resource asks for v1. Restart the kernel and run from the top. The helper reports whether the remaining mismatch is tenant (`tid`), destination API (`aud`), or child caller (`azp`), without printing the token.

## 4. Test allowed and forbidden calls

**Setup needed:** the token cell must have succeeded. No additional service setup.

The helper starts the local MCP server, tries an anonymous call, then calls with our real token. The server verifies the signature, tenant, target API, caller, expiry, and permission. It stops when this cell finishes.

In [ ]:
await explore_mcp_identity(settings, access)

**Read the actual output:**

| Attempt | Expected result | Meaning |
|---|---|---|
| No token | HTTP 401 | No badge: no entry |
| Child token + `Workshop.Read.Application` role | `who_am_i` returns the child client ID | Entra signed the badge; the server accepted its key |
| Agent-only call to `user_context` | Denied; hidden from tool discovery too | A badge is not permission to act as a user |

These checks happen **before any model runs**. A model cannot talk its way past them. A token without the required role is rejected with HTTP 403; we do not fabricate such a token for the live demo.

**Where to see identity evidence:** Entra > Agents > Agent identities > your child > Sign-in logs. These are authentication events, not model/tool traces. Token reuse can mean there is not a new sign-in event for every tool call.

**Say:** "Our worker has a real badge, and the tool checks its key. Now we let an actual agent use this protected tool."

Next: [02 - Run the agent and see Azure traces](./02_Agent_and_Azure_Traces.ipynb).